In [10]:
!pip -q install pandas openpyxl scikit-learn xgboost joblib

import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import files

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

print("Libraries installed and imported successfully.")


Libraries installed and imported successfully.


In [11]:

# Upload the Excel file
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

excel_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".xlsx", ".xls"))
]

if len(excel_files) != 1:
    raise ValueError("Please upload exactly one Excel file.")

DATA_PATH = Path("/content") / excel_files[0]

# General settings
RANDOM_STATE = 20261107
TARGET = "delta_vwc"
SPLIT_COL = "split"
SHEET = "VWC_Response"
DICT_SHEET = "Data_Dictionary"

MODEL_OUT = Path("/content/rayyan_delta_vwc_model.joblib")

EVAL_SPLITS = ["test", "stress_test"]

# Read Excel sheets
excel = pd.ExcelFile(DATA_PATH)

required_sheets = [SHEET, DICT_SHEET]

for sheet in required_sheets:
    if sheet not in excel.sheet_names:
        raise ValueError(f"Missing Excel sheet: {sheet}")

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
dictionary = pd.read_excel(DATA_PATH, sheet_name=DICT_SHEET)

# Validate required columns
required_data_cols = [
    "response_eligible",
    TARGET,
    SPLIT_COL
]

for col in required_data_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column in VWC_Response: {col}")

required_dict_cols = ["sheet", "role", "column"]

for col in required_dict_cols:
    if col not in dictionary.columns:
        raise ValueError(f"Missing column in Data_Dictionary: {col}")

# Extract features from Data Dictionary
mask = (
    dictionary["sheet"].eq(SHEET)
    & dictionary["role"].eq("feature")
)

features = dictionary.loc[mask, "column"].dropna().tolist()

if not features:
    raise ValueError("No features were found in Data_Dictionary.")

missing_features = [col for col in features if col not in df.columns]

if missing_features:
    raise ValueError(f"Features missing from dataset: {missing_features}")

# Keep eligible records with valid target
eligible = df["response_eligible"]

if not pd.api.types.is_bool_dtype(eligible):
    eligible = (
        eligible.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

before = len(df)

df = df[
    eligible
    & df[TARGET].notna()
].copy()

df[SPLIT_COL] = df[SPLIT_COL].astype(str).str.strip().str.lower()

df = df.reset_index(drop=True)

print("Data loaded successfully!")
print("--------------------------------")
print("File:", DATA_PATH.name)
print("Valid records:", len(df))
print("Excluded records:", before - len(df))
print("Number of features:", len(features))
print("\nFeature columns:")
print(features)

print("\nSplit distribution:")
print(df[SPLIT_COL].value_counts().to_dict())

print("\nMissing values:")
print(df[features].isna().sum())


Saving Rayyan_Bermuda_Model_Training_2024_2025.xlsx to Rayyan_Bermuda_Model_Training_2024_2025 (2).xlsx
Data loaded successfully!
--------------------------------
File: Rayyan_Bermuda_Model_Training_2024_2025 (2).xlsx
Valid records: 1272
Excluded records: 1
Number of features: 23

Feature columns:
['pre_vwc', 'sensor_age_min', 'sensor_valid', 'pending_or_active', 'hours_since_irrigation', 'trend_vwc_h', 'source_remaining_l', 'unreserved_remaining_l', 'priority_level', 'other_zone_pre_vwc', 'other_zone_sensor_valid', 'other_zone_priority_level', 'forecast_rain_3h_mm', 'forecast_et0_3h_mm', 'forecast_et0_12h_mm', 'area_m2', 'root_depth_m', 'plant_demand_factor', 'doy_sin', 'doy_cos', 'hour_local', 'planned_volume_l', 'planned_duration_min']

Split distribution:
{'train': 601, 'test': 305, 'validation': 264, 'stress_test': 102}

Missing values:
pre_vwc                      0
sensor_age_min               0
sensor_valid                 0
pending_or_active            0
hours_since_irrigation

In [12]:

# --------------------------------------------------
# 1. Prepare X and y
# --------------------------------------------------

def prepare_xy(df, features, split_name):

    part = df[df[SPLIT_COL] == split_name].copy()

    if part.empty:
        raise ValueError(
            f"No eligible records found for split: {split_name}"
        )

    X = part[features].copy()

    # Convert boolean features to numeric values
    for col in features:

        if pd.api.types.is_bool_dtype(X[col]):
            X[col] = X[col].astype(int)

        try:
            X[col] = pd.to_numeric(
                X[col], errors="raise"
            ).astype(float)

        except (ValueError, TypeError):
            raise ValueError(
                f"Feature '{col}' contains non-numeric values."
            )

    y = part[TARGET].astype(float)

    return X, y


# --------------------------------------------------
# 2. Build preprocessing pipeline
# --------------------------------------------------

def build_pipeline(estimator):

    return Pipeline([
        (
            "impute",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        ("model", estimator)
    ])


# --------------------------------------------------
# 3. Define candidate models
# --------------------------------------------------

def get_candidates():

    candidates = {

        "RandomForest": (

            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            {
                "model__n_estimators": [200, 400, 600],
                "model__max_depth": [None, 6, 8, 12],
                "model__min_samples_leaf": [1, 3, 5, 10],
                "model__max_features": ["sqrt", 0.5, 1.0],
            }
        ),

        "XGBoost": (

            XGBRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                objective="reg:squarederror",
                verbosity=0
            ),

            {
                "model__n_estimators": [200, 400, 800],
                "model__learning_rate": [0.02, 0.05, 0.1],
                "model__max_depth": [2, 3, 4, 6],
                "model__subsample": [0.7, 0.85, 1.0],
                "model__colsample_bytree": [0.6, 0.8, 1.0],
                "model__min_child_weight": [1, 3, 5],
            }
        )
    }

    return candidates


# --------------------------------------------------
# 4. Tune models using train and validation
# --------------------------------------------------

def tune_model(
    name,
    estimator,
    param_grid,
    X_train,
    y_train,
    X_val,
    y_val
):

    X_all = pd.concat([X_train, X_val])
    y_all = pd.concat([y_train, y_val])

    # -1 = training
    #  0 = validation

    fold = np.r_[
        np.full(len(X_train), -1),
        np.zeros(len(X_val), dtype=int)
    ]

    search = RandomizedSearchCV(

        build_pipeline(estimator),

        param_grid,

        n_iter=25,

        cv=PredefinedSplit(fold),

        scoring="neg_root_mean_squared_error",

        random_state=RANDOM_STATE,

        n_jobs=-1,

        refit=True
    )

    search.fit(X_all, y_all)

    print(
        f"{name}: Best validation RMSE = "
        f"{-search.best_score_:.5f}"
    )

    return search.best_estimator_, -search.best_score_


# --------------------------------------------------
# 5. Evaluate models
# --------------------------------------------------

def evaluate(model, X, y):

    pred = model.predict(X)

    return {

        "RMSE": np.sqrt(
            mean_squared_error(y, pred)
        ),

        "MAE": mean_absolute_error(y, pred),

        "R2": (
            r2_score(y, pred)
            if len(y) >= 2
            else np.nan
        )
    }


# --------------------------------------------------
# 6. Feature importance
# --------------------------------------------------

def print_feature_importance(model, top_n=10):

    names = (
        model.named_steps["impute"]
        .get_feature_names_out()
    )

    importances = (
        model.named_steps["model"]
        .feature_importances_
    )

    importance_df = pd.Series(
        importances,
        index=names
    ).sort_values(ascending=False).head(top_n)

    print("\nTop Important Features:")
    print("--------------------------------")

    for feature, value in importance_df.items():

        print(f"{feature:<30} {value:.4f}")


In [14]:

# --------------------------------------------------
# 1. Prepare datasets
# --------------------------------------------------

X_train, y_train = prepare_xy(df, features, "train")

X_val, y_val = prepare_xy(df, features, "validation")

eval_sets = {
    split: prepare_xy(df, features, split)
    for split in EVAL_SPLITS
}

print("[1] Dataset sizes:")
print("Training:", len(X_train))
print("Validation:", len(X_val))

for split, (X, y) in eval_sets.items():
    print(f"{split}: {len(X)}")


# --------------------------------------------------
# 2. Baseline model
# --------------------------------------------------

X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

baseline = build_pipeline(
    DummyRegressor()
)

baseline.fit(X_train_val, y_train_val)

trained = {
    "Baseline (Mean)": baseline
}


# --------------------------------------------------
# 3. Train and tune candidate models
# --------------------------------------------------

print("\n[2] Training and tuning models...")

val_scores = {}

for name, (estimator, param_grid) in get_candidates().items():

    model, score = tune_model(
        name,
        estimator,
        param_grid,
        X_train,
        y_train,
        X_val,
        y_val
    )

    trained[name] = model
    val_scores[name] = score


# --------------------------------------------------
# 4. Evaluate on test and stress_test
# --------------------------------------------------

print("\n[3] Final Evaluation")

for split_name, (X, y) in eval_sets.items():

    rows = []

    for name, model in trained.items():

        result = evaluate(model, X, y)

        rows.append({
            "Model": name,
            **result
        })

    results_df = (
        pd.DataFrame(rows)
        .set_index("Model")
    )

    print(f"\n--- {split_name} ({len(y)} records) ---")

    display(
        results_df.style.format("{:.5f}")
    )


# --------------------------------------------------
# 5. Select the best model using validation only
# --------------------------------------------------

best_name = min(
    val_scores,
    key=val_scores.get
)

best_model = trained[best_name]

print("\n[4] Selected Model:")
print(best_name)

print("\nBest Validation RMSE:")
print(f"{val_scores[best_name]:.5f}")


# --------------------------------------------------
# 6. Feature importance
# --------------------------------------------------

print_feature_importance(
    best_model,
    top_n=10
)


# --------------------------------------------------
# 7. Save the model
# --------------------------------------------------

model_bundle = {
    "model": best_model,
    "features": features
}

joblib.dump(
    model_bundle,
    MODEL_OUT
)

print("\n[5] Model saved successfully!")
print(MODEL_OUT)


# --------------------------------------------------
# 8. Download the trained model
# --------------------------------------------------

print("\nDownloading your trained model...")

files.download(
    str(MODEL_OUT)
)

print("\nRayyan training completed successfully!")


[1] Dataset sizes:
Training: 601
Validation: 264
test: 305
stress_test: 102

[2] Training and tuning models...
RandomForest: Best validation RMSE = 0.00155
XGBoost: Best validation RMSE = 0.00155

[3] Final Evaluation

--- test (305 records) ---


,RMSE,MAE,R2
Model,,,
Baseline (Mean),0.00913,0.00633,-0.00077
RandomForest,0.00146,0.00117,0.97452
XGBoost,0.00143,0.00114,0.97553



--- stress_test (102 records) ---


,RMSE,MAE,R2
Model,,,
Baseline (Mean),0.01048,0.00752,-0.02413
RandomForest,0.00331,0.00155,0.89784
XGBoost,0.00329,0.00148,0.89893



[4] Selected Model:
XGBoost

Best Validation RMSE:
0.00155

Top Important Features:
--------------------------------
unreserved_remaining_l         0.5380
source_remaining_l             0.2984
planned_volume_l               0.1162
forecast_et0_3h_mm             0.0190
pre_vwc                        0.0159
hours_since_irrigation         0.0030
hour_local                     0.0023
trend_vwc_h                    0.0017
doy_cos                        0.0009
other_zone_pre_vwc             0.0009

[5] Model saved successfully!
/content/rayyan_delta_vwc_model.joblib



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Rayyan training completed successfully!
